# Lab 7 — Capstone: Serve a Local RAG Agent Behind an API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/humzaahmad906/applied-ml-academy/blob/main/content/vlm-guide/notebooks/16_lab_capstone.ipynb)

Wire a FAISS retrieval index and a ReAct agent together: a `retrieve_context` tool backed by a vector index, plus calculator and datetime tools. This notebook runs the full agent + RAG loop in-cell; the FastAPI serving layer is written to a script file since Colab can't cleanly host a long-running server in a notebook cell.

Runs on Google Colab — for the GPU labs, set Runtime → Change runtime type → GPU.

The full write-up (FastAPI `lifespan`, Pydantic schemas, `TestClient` tests, vLLM/Ollama backends) lives in the [lesson](../16_lab_capstone.md).

In [ ]:
!pip install -q transformers accelerate torch sentence-transformers faiss-cpu fastapi uvicorn httpx pydantic


## The RAG index

Recap from Lab 5 — `IndexFlatIP` over L2-normalized embeddings = cosine search. In production swap to `faiss.read_index` on a pre-built file; the index here builds in under a second.

**Models:** `all-MiniLM-L6-v2` (~90 MB) for embeddings + `Qwen/Qwen2.5-1.5B-Instruct` (~3.1 GB bf16) for generation. Swap to `Qwen/Qwen2.5-0.5B-Instruct` (~1 GB) if VRAM is tight.

In [ ]:
import re, math, datetime, logging, random

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s - %(message)s")
logger = logging.getLogger(__name__)

_DOCUMENTS = [
    "The Eiffel Tower stands 330 m (1,083 ft) tall, located in Paris, France.",
    "Python is a high-level programming language created by Guido van Rossum in 1991.",
    "The Transformer architecture uses self-attention and feed-forward blocks (Vaswani 2017).",
    "FAISS is Facebook AI's library for efficient similarity search on dense vectors.",
    "RAG (Retrieval-Augmented Generation) grounds LLM outputs with retrieved external evidence.",
    "FastAPI is a modern Python web framework built on Starlette and Pydantic with OpenAPI docs.",
]


def build_faiss_index(docs: list[str], embed_model: SentenceTransformer):
    embs = embed_model.encode(docs, normalize_embeddings=True, show_progress_bar=False)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs.astype(np.float32))
    return index, docs

def _retrieve(query: str, index, chunks: list[str], embed_model, top_k: int = 3) -> str:
    q = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    _, ids = index.search(q, top_k)
    return "\n".join(f"[{i+1}] {chunks[idx]}" for i, idx in enumerate(ids[0]) if idx >= 0)


## Tools

`make_retrieve_context` is a closure that binds the index once, keeping the tool signature as `(query: str) -> str` — the same shape as `calculator` and `get_datetime`.

In [ ]:
def make_retrieve_context(index, chunks: list[str], embed_model) -> callable:
    """Closure — binds the index once; keeps the tool signature as (query: str) -> str."""
    return lambda query: _retrieve(query, index, chunks, embed_model)


def calculator(expression: str) -> str:
    safe = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    safe["__builtins__"] = {}
    try:
        return str(eval(expression, safe))  # noqa: S307
    except Exception as exc:
        return f"Error: {exc}"

def get_datetime() -> str:
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")


## The agent

The Lab 6 ReAct loop, refactored to receive models + registry as arguments instead of module globals — this is the shape the FastAPI service below plugs into. System prompt delta: `retrieve_context` is listed first and the agent is told to use it before answering factual questions.

In [ ]:
_SYSTEM = """\
You answer questions using tools. At each turn emit EXACTLY ONE of:

  Thought: <reasoning>
  Action: tool_name(argument)
  Answer: <final answer>

Available tools:
  retrieve_context(query)  – search the local knowledge base for relevant facts
  calculator(expression)   – evaluate a Python math expression
  get_datetime()           – return the current date and time

Use retrieve_context for any factual question before answering.
"""

_ACTION_RE = re.compile(r"Action:\s*(\w+)\(([^)]*)\)")
_ANSWER_RE = re.compile(r"Answer:\s*(.+)", re.DOTALL)


def _run_agent(
    question: str, registry: dict, tokenizer, model, device: str, max_steps: int = 8
) -> str:
    def _chat(msgs: list[dict]) -> str:
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    messages = [{"role": "system", "content": _SYSTEM}, {"role": "user", "content": question}]
    for _ in range(max_steps):
        resp = _chat(messages)
        m = _ANSWER_RE.search(resp)
        if m:
            return m.group(1).strip()
        m = _ACTION_RE.search(resp)
        if m:
            name, raw = m.group(1), m.group(2).strip()
            args = [a.strip().strip("\"'") for a in raw.split(",")] if raw else []
            if name not in registry:
                obs = f"Unknown tool '{name}'. Available: {list(registry)}"
            else:
                try:
                    obs = registry[name](*args)
                except Exception as exc:
                    obs = f"Tool error: {exc}"
            messages += [{"role": "assistant", "content": resp},
                         {"role": "user",      "content": f"Observation: {obs}"}]
        else:
            messages += [{"role": "assistant", "content": resp},
                         {"role": "user", "content": "Continue. Emit Action: or Answer:."}]
    return "Agent budget exhausted without a final answer."


## Load models and build the index

This stands in for the FastAPI `lifespan` hook below: load the embedding model, build the FAISS index, load the generation model, and assemble the tool registry once.

In [ ]:
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {device}")

embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
index, chunks = build_faiss_index(_DOCUMENTS, embed_model)
print(f"FAISS index: {len(chunks)} docs")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # swap to Qwen/Qwen2.5-0.5B-Instruct if VRAM is tight
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
gen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map=device
)
gen_model.eval()

registry = {
    "retrieve_context": make_retrieve_context(index, chunks, embed_model),
    "calculator":       calculator,
    "get_datetime":     get_datetime,
}

for q in ["How tall is the Eiffel Tower in feet? (1 m = 3.28084 ft)",
          "What is 6 * 7?",
          "Who created Python and when?"]:
    print(f"\n{'='*50}\nQ: {q}\n{'='*50}")
    print(f">>> {_run_agent(q, registry, tokenizer, gen_model, device)}\n")


## Serving it behind FastAPI

The routes, `lifespan` startup hook, and Pydantic schemas below are the production shape of the loop above — `_run_agent` is unchanged, it just receives its models from app state instead of notebook globals. Colab can't host a long-running server in a cell without blocking the notebook, so this cell writes the service to a script file instead of running it.

In [ ]:
%%writefile rag_agent_service.py
"""Standalone script — run with: uvicorn rag_agent_service:app --host 0.0.0.0 --port 8000"""
import re, math, datetime, logging
from contextlib import asynccontextmanager

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s - %(message)s")
logger = logging.getLogger(__name__)

_DOCUMENTS = [
    "The Eiffel Tower stands 330 m (1,083 ft) tall, located in Paris, France.",
    "Python is a high-level programming language created by Guido van Rossum in 1991.",
    "The Transformer architecture uses self-attention and feed-forward blocks (Vaswani 2017).",
    "FAISS is Facebook AI's library for efficient similarity search on dense vectors.",
    "RAG (Retrieval-Augmented Generation) grounds LLM outputs with retrieved external evidence.",
    "FastAPI is a modern Python web framework built on Starlette and Pydantic with OpenAPI docs.",
]


def build_faiss_index(docs, embed_model):
    embs = embed_model.encode(docs, normalize_embeddings=True, show_progress_bar=False)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs.astype(np.float32))
    return index, docs

def _retrieve(query, index, chunks, embed_model, top_k=3):
    q = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    _, ids = index.search(q, top_k)
    return "\n".join(f"[{i+1}] {chunks[idx]}" for i, idx in enumerate(ids[0]) if idx >= 0)

def make_retrieve_context(index, chunks, embed_model):
    return lambda query: _retrieve(query, index, chunks, embed_model)

def calculator(expression: str) -> str:
    safe = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    safe["__builtins__"] = {}
    try:
        return str(eval(expression, safe))  # noqa: S307
    except Exception as exc:
        return f"Error: {exc}"

def get_datetime() -> str:
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

_SYSTEM = """\
You answer questions using tools. At each turn emit EXACTLY ONE of:

  Thought: <reasoning>
  Action: tool_name(argument)
  Answer: <final answer>

Available tools:
  retrieve_context(query)  – search the local knowledge base for relevant facts
  calculator(expression)   – evaluate a Python math expression
  get_datetime()           – return the current date and time

Use retrieve_context for any factual question before answering.
"""

_ACTION_RE = re.compile(r"Action:\s*(\w+)\(([^)]*)\)")
_ANSWER_RE = re.compile(r"Answer:\s*(.+)", re.DOTALL)


def _run_agent(question, registry, tokenizer, model, device, max_steps=8):
    def _chat(msgs):
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    messages = [{"role": "system", "content": _SYSTEM}, {"role": "user", "content": question}]
    for _ in range(max_steps):
        resp = _chat(messages)
        m = _ANSWER_RE.search(resp)
        if m:
            return m.group(1).strip()
        m = _ACTION_RE.search(resp)
        if m:
            name, raw = m.group(1), m.group(2).strip()
            args = [a.strip().strip("\"'") for a in raw.split(",")] if raw else []
            if name not in registry:
                obs = f"Unknown tool '{name}'. Available: {list(registry)}"
            else:
                try:
                    obs = registry[name](*args)
                except Exception as exc:
                    obs = f"Tool error: {exc}"
            messages += [{"role": "assistant", "content": resp},
                         {"role": "user",      "content": f"Observation: {obs}"}]
        else:
            messages += [{"role": "assistant", "content": resp},
                         {"role": "user", "content": "Continue. Emit Action: or Answer:."}]
    return "Agent budget exhausted without a final answer."

_state: dict = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    device = ("cuda" if torch.cuda.is_available()
              else "mps" if torch.backends.mps.is_available() else "cpu")
    logger.info("startup: device=%s", device)

    embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
    index, chunks = build_faiss_index(_DOCUMENTS, embed_model)
    logger.info("FAISS index: %d docs", len(chunks))

    tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
    gen = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct", torch_dtype=torch.bfloat16, device_map=device
    )
    gen.eval()
    logger.info("generation model ready")

    _state.update({
        "tokenizer": tok, "gen_model": gen, "device": device,
        "registry": {
            "retrieve_context": make_retrieve_context(index, chunks, embed_model),
            "calculator":       calculator,
            "get_datetime":     get_datetime,
        },
    })
    logger.info("service ready")
    yield
    _state.clear()


app = FastAPI(title="RAG Agent API", version="1.0.0", lifespan=lifespan)


class ChatRequest(BaseModel):
    question: str = Field(..., min_length=1, max_length=1000)
    max_steps: int = Field(default=8, ge=1, le=20)

class ChatResponse(BaseModel):
    question: str
    answer: str

class HealthResponse(BaseModel):
    status: str
    models_loaded: bool


@app.get("/health", response_model=HealthResponse)
async def health() -> HealthResponse:
    return HealthResponse(status="ok", models_loaded=bool(_state))


@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest) -> ChatResponse:
    if not _state:
        raise HTTPException(status_code=503, detail="Models still loading.")
    try:
        answer = _run_agent(
            req.question, _state["registry"],
            _state["tokenizer"], _state["gen_model"], _state["device"],
            req.max_steps,
        )
    except Exception as exc:
        logger.exception("agent error: %s", req.question)
        raise HTTPException(status_code=500, detail=str(exc)) from exc
    return ChatResponse(question=req.question, answer=answer)


## Run the service

Run the script above from a terminal (not from this notebook — the server call blocks forever, which would hang a Colab cell):

```bash
uvicorn rag_agent_service:app --host 0.0.0.0 --port 8000
```

Then exercise it:

```bash
curl http://localhost:8000/health

curl -s -X POST http://localhost:8000/chat \
     -H "Content-Type: application/json" \
     -d '{"question": "How tall is the Eiffel Tower in feet?"}' | python3 -m json.tool
```

The full `TestClient` test suite and validation checks are in the lesson.

## Stacks & alternatives

The FastAPI routes, Pydantic schemas, and tool registry above are unchanged if you swap generation backends — only the `_chat()` call inside `_run_agent` changes:

- **vLLM** — `vllm serve Qwen/Qwen2.5-1.5B-Instruct --port 11434 --dtype bfloat16`; production throughput via PagedAttention, 5-10x a naive HF `generate` loop.
- **Ollama** — `ollama pull qwen2.5:1.5b && ollama serve`; easiest local dev, GGUF quantized (~800 MB vs 3.1 GB bf16), macOS Metal support.

Both expose an OpenAI-compatible `/v1/chat/completions` endpoint — swapping is one URL string and one model name. The lesson also covers streaming (`POST /chat/stream`), multi-turn memory, an eval harness across all three backends, and containerizing with Docker.

See [16_lab_capstone.md](../16_lab_capstone.md) for the code and tradeoffs of each.